# H8.1 — EDA поведения модели и harness

Идея 1 - данные не до конца честные. Присутствуют зависимости в датасете, которые можно выявить.

Идея 2 - модель не глупая. Но думать ей лениво, вопрос какие рассуждения она делает



In [1]:
from pathlib import Path
import json, zipfile, collections, re
import pandas as pd

## 0. INPUTS

In [ ]:

DATA_DIR = Path('.')
EDA_JSON = DATA_DIR / 'eda_results/eda_tables.json'
RUN_ZIP = DATA_DIR / 'h8_24_regression_model.zip'
OUTPUT_XLSX = DATA_DIR / 'H8_1_behavior_EDA_review.xlsx'

def normalize_archive_path(value):
    if not value:
        return value
    parts = str(value).replace('\\\\', '/').lstrip('./').split('/')
    return '/'.join(parts[1:]) if len(parts) > 1 else parts[0]

with EDA_JSON.open(encoding='utf-8') as f:
    eda = json.load(f)
package_df = pd.DataFrame(eda['tables']['package_level'])
evidence_df = pd.DataFrame(eda['tables']['evidence_details'])

# Keep the original fixed sample order, but never use sample_id as a predictor.
package_df = package_df.sort_values('sample_id').reset_index(drop=True)

## 1. LOAD PER-PACKAGE HARNESS ARTIFACTS AND API TRACE

In [ ]:

with zipfile.ZipFile(RUN_ZIP) as zf:
    names = set(zf.namelist())
    metrics = json.loads(zf.read('metrics.json'))
    config = json.loads(zf.read('config.json'))
    trace_rows = [json.loads(line) for line in zf.read('api_trace.jsonl').decode('utf-8').splitlines() if line.strip()]

    metric_by_archive = {p['archive']: p for p in metrics.get('packages', [])}
    artifacts = {}
    for row in package_df.to_dict('records'):
        sid = int(row['sample_id'])
        prefix = f'chats/{sid:04d}'
        def read_json(suffix):
            path = f'{prefix}.{suffix}'
            return json.loads(zf.read(path)) if path in names else {}
        dossier_path = f'{prefix}.dossier.txt'
        dossier = zf.read(dossier_path).decode('utf-8', errors='replace') if dossier_path in names else ''
        artifacts[sid] = {
            'behavior_map': read_json('behavior_map.json'),
            'static_analysis': read_json('static_analysis.json'),
            'dossier': dossier,
            'chat': read_json('json'),
            'metrics': metric_by_archive.get(row['archive'], {}),
        }

## 2. NORMALIZE TRACE INTO PACKAGE-LEVEL RESOURCE METRICS

In [ ]:
trace_by_sid = collections.defaultdict(list)
for ev in trace_rows:
    sid = (ev.get('sample') or {}).get('sample_id')
    if sid is not None:
        trace_by_sid[int(sid)].append(ev)

nav_rows = []
for row in package_df.to_dict('records'):
    sid = int(row['sample_id'])
    events = trace_by_sid[sid]
    requests = [e for e in events if 'request_id' in e]
    tool_events = [e for e in events if e.get('event') == 'tool_call']
    retries = [e for e in events if e.get('event') == 'harness_retry']
    usage = [e.get('response', {}).get('usage', {}) for e in requests]
    static = artifacts[sid]['static_analysis']
    metric = artifacts[sid]['metrics']
    tool_counts = collections.Counter(e.get('tool', 'unknown') for e in tool_events)
    tool_outcomes = collections.Counter(f"{e.get('tool','unknown')}:{e.get('outcome','unknown')}" for e in tool_events)
    nav_rows.append({
        'sample_id': sid, 'archive': row['archive'], 'ecosystem': row['ecosystem'],
        'outcome': row['outcome'], 'status': row['status'],
        'turns_from_eda': row.get('turns'),
        'api_requests': len(requests), 'tool_calls_trace': len(tool_events),
        'tool_errors': sum(1 for e in tool_events if e.get('outcome') in {'error','read_failed','invalid_arguments','path_not_found','path_is_file','offset_out_of_range'}),
        'read_file_calls': tool_counts.get('read_file', 0),
        'read_file_failed': sum(1 for e in tool_events if e.get('tool') == 'read_file' and e.get('outcome') in {'read_failed','error'}),
        'get_behavior_map_calls': tool_counts.get('get_behavior_map', 0),
        'submit_analysis_calls': tool_counts.get('submit_analysis', 0),
        'submit_verdict_calls': tool_counts.get('submit_verdict', 0),
        'no_think_retries_trace': len(retries),
        'prompt_tokens': sum(u.get('prompt_tokens', 0) or 0 for u in usage),
        'completion_tokens': sum(u.get('completion_tokens', 0) or 0 for u in usage),
        'reasoning_tokens': sum((u.get('completion_tokens_details') or {}).get('reasoning_tokens', 0) or 0 for u in usage),
        'total_tokens': sum(u.get('total_tokens', 0) or 0 for u in usage),
        'trace_elapsed_ms': round(sum(e.get('elapsed_ms', 0) or 0 for e in requests), 1),
        'scanned_bytes': static.get('scanned_bytes', row.get('scanned_bytes')),
        'scanned_text_files': static.get('scanned_text_files', row.get('scanned_text_files')),
        'dossier_chars_metrics': metric.get('dossier_chars', row.get('dossier_chars')),
        'dossier_files_count_metrics': len(metric.get('dossier_files', [])) if metric else row.get('dossier_files_count'),
        'tool_outcomes_compact': '; '.join(f'{k}={v}' for k,v in sorted(tool_outcomes.items())),
    })
navigation_df = pd.DataFrame(nav_rows)

## 3. COVERAGE: SCANNED / MODEL-VISIBLE / BEHAVIOR-MAP / DOSSIER PATHS

In [ ]:

coverage_rows = []
coverage_fact_rows = []
for row in package_df.to_dict('records'):
    sid = int(row['sample_id'])
    art = artifacts[sid]
    bm, sa, metric = art['behavior_map'], art['static_analysis'], art['metrics']
    visible = set(sa.get('model_visible_paths', []))
    scanned = set(sa.get('scanned_paths', []))
    dossier_files = metric.get('dossier_files', [])
    dossier_norm = {normalize_archive_path(p) for p in dossier_files}
    groups = bm.get('groups', {})
    group_counts = {g: len(groups.get(g, []) or []) for g in ['triggers','sources','transforms','actions','destinations','other']}
    registry = bm.get('evidence_registry', {}) or {}
    registry_items = []
    if isinstance(registry, dict):
        for eid, item in registry.items():
            if isinstance(item, dict):
                x = dict(item); x.setdefault('evidence_id', eid); registry_items.append(x)
    elif isinstance(registry, list):
        registry_items = registry
    for group, items in groups.items():
        for item in items or []:
            path = item.get('path')
            eid = item.get('evidence_id')
            in_visible = path in visible if path else None
            in_dossier = path in dossier_norm if path else None
            coverage_fact_rows.append({
                'sample_id': sid, 'archive': row['archive'], 'outcome': row['outcome'],
                'group': group, 'evidence_id': eid, 'path': path, 'line': item.get('line'),
                'kind': item.get('kind'), 'detail': item.get('detail'),
                'source_file_scanned': path in scanned if path else None,
                'source_file_model_visible': in_visible,
                'source_file_in_dossier_manifest': in_dossier,
                'exposed_via_behavior_map': True,
                'selected_by_model': None,
            })
    # Attach model selection status from evidence_details where available.
    ev_by_id = {(int(x['sample_id']), str(x['evidence_id'])): bool(x.get('selected_by_model')) for x in evidence_df.to_dict('records')}
    for fact in coverage_fact_rows:
        if fact['sample_id'] == sid and fact['evidence_id'] is not None:
            fact['selected_by_model'] = ev_by_id.get((sid, str(fact['evidence_id'])))
    chain_list = bm.get('candidate_chains', []) or []
    full_co_presence = all(group_counts[g] > 0 for g in ['sources','transforms','actions','destinations'])
    chain_stages = [set(c.get('observed_stages', [])) for c in chain_list if isinstance(c, dict)]
    has_explicit_full_candidate = any({'sources','transforms','actions','destinations'}.issubset(stages) for stages in chain_stages)
    coverage_rows.append({
        'sample_id': sid, 'archive': row['archive'], 'ecosystem': row['ecosystem'],
        'true_label': row['true_label'], 'predicted_label': row['predicted_label'], 'outcome': row['outcome'],
        'scanned_files_count': len(scanned), 'model_visible_files_count': len(visible),
        'dossier_manifest_files_count': len(dossier_norm),
        'scanned_not_model_visible_count': len(scanned - visible),
        'map_evidence_total': sum(group_counts.values()), **{f'map_{k}':v for k,v in group_counts.items()},
        'map_candidate_chains': len(chain_list),
        'map_evidence_source_file_in_dossier_count': sum(1 for x in coverage_fact_rows if x['sample_id']==sid and x['source_file_in_dossier_manifest'] is True),
        'map_evidence_source_file_outside_dossier_count': sum(1 for x in coverage_fact_rows if x['sample_id']==sid and x['source_file_in_dossier_manifest'] is False),
        'all_four_groups_present_cooccurrence': full_co_presence,
        'explicit_candidate_chain_has_all_four_stages': has_explicit_full_candidate,
        'prompt_injection_candidates_telemetry': sa.get('prompt_injection_candidate_count', 0),
    })
coverage_df = pd.DataFrame(coverage_rows)
coverage_facts_df = pd.DataFrame(coverage_fact_rows)

## 4. UTILIZATION: WHICH AVAILABLE EVIDENCE IDS DID THE MODEL SELECT?

In [ ]:

utilization_df = evidence_df.copy()
utilization_df['selected_by_model'] = utilization_df['selected_by_model'].fillna(False).astype(bool)
util_summary = (utilization_df.groupby(['outcome','group'], dropna=False)
                .agg(available=('evidence_id','count'), selected=('selected_by_model','sum'))
                .reset_index())
util_summary['selection_rate'] = util_summary['selected'] / util_summary['available'].replace(0, pd.NA)
util_summary['not_selected'] = util_summary['available'] - util_summary['selected']
util_by_package = (utilization_df.groupby(['sample_id','archive','outcome','group'])
                   .agg(available=('evidence_id','count'), selected=('selected_by_model','sum'))
                   .reset_index())
util_by_package['selection_rate'] = util_by_package['selected'] / util_by_package['available'].replace(0, pd.NA)

## 5. CHAIN RECONSTRUCTION: distinguish simple group co-presence from a

In [ ]:

chain_rows = []
for row in package_df.to_dict('records'):
    sid = int(row['sample_id']); bm = artifacts[sid]['behavior_map']
    groups = bm.get('groups', {})
    counts = {g: len(groups.get(g, []) or []) for g in ['triggers','sources','transforms','actions','destinations']}
    chains = bm.get('candidate_chains', []) or []
    stages_union = set()
    chain_descriptions = []
    for c in chains:
        if isinstance(c, dict):
            stages = c.get('observed_stages', []) or []
            stages_union.update(stages)
            chain_descriptions.append(f"{','.join(stages)} [{c.get('relationship_status','unspecified')}]")
    chain_rows.append({
        'sample_id': sid, 'archive': row['archive'], 'outcome': row['outcome'],
        **{f'{g}_count': counts[g] for g in counts},
        'candidate_chain_count': len(chains),
        'candidate_chain_stage_union': ' → '.join(g for g in ['triggers','sources','transforms','actions','destinations'] if g in stages_union),
        'has_source_transform_action_destination': all(counts[g] > 0 for g in ['sources','transforms','actions','destinations']),
        'candidate_chain_has_all_four_stages': any(isinstance(c,dict) and {'sources','transforms','actions','destinations'}.issubset(set(c.get('observed_stages',[]))) for c in chains),
        'candidate_chain_relationship_status': ' | '.join(chain_descriptions),
        'model_selected_any_evidence': bool((utilization_df[utilization_df.sample_id == sid]['selected_by_model']).any()),
        'model_selected_all_evidence': bool(len(utilization_df[utilization_df.sample_id == sid]) and utilization_df[utilization_df.sample_id == sid]['selected_by_model'].all()),
    })
chain_df = pd.DataFrame(chain_rows)

## 6. FAILURE TAXONOMY: create review prompts, not automatic ground truth.

In [ ]:
all_review = package_df[package_df['outcome'].isin(['FN','FP'])].copy()
review_rows = []
for row in all_review.to_dict('records'):
    sid = int(row['sample_id'])
    cov = coverage_df[coverage_df.sample_id == sid].iloc[0]
    ch = chain_df[chain_df.sample_id == sid].iloc[0]
    nav = navigation_df[navigation_df.sample_id == sid].iloc[0]
    ev = utilization_df[utilization_df.sample_id == sid]
    selected_n = int(ev['selected_by_model'].sum()) if len(ev) else 0
    available_n = int(len(ev))
    selected_rate = selected_n / available_n if available_n else None
    if row['status'] != 'completed':
        candidate = 'протокольный отказ / анализ не завершён'
        rationale = 'Статус пакета не completed; сначала проверить trace и ошибки инструмента.'
    elif row['outcome'] == 'FN' and nav['read_file_failed'] > 0:
        candidate = 'навигация / неверный путь чтения'
        rationale = 'В trace есть неуспешный read_file; проверить, был ли нужный файл доступен другим каналом.'
    elif row['outcome'] == 'FN' and ch['has_source_transform_action_destination'] and selected_rate is not None and selected_rate < 0.5:
        candidate = 'недостаточное использование / связывание evidence'
        rationale = 'Есть совместное присутствие четырёх групп, но доля выбранных evidence ниже 50%; это диагностическая эвристика.'
    elif row['outcome'] == 'FN' and selected_n > 0:
        candidate = 'возможная неверная интерпретация найденных фактов'
        rationale = 'Модель выбрала часть evidence, но итоговый вердикт benign; требуется ручная проверка reasoning и контекста.'
    elif row['outcome'] == 'FP':
        candidate = 'возможная гиперинтерпретация benign-поведения'
        rationale = 'Ложноположительный вердикт; проверить, какие API/цепочки модель сочла подозрительными и почему.'
    else:
        candidate = 'недостаточно данных для диагноза'
        rationale = 'Нужен ручной разбор dossier, behavior map, selected evidence и полного диалога.'
    review_rows.append({
        'sample_id': sid, 'archive': row['archive'], 'ecosystem': row['ecosystem'],
        'true_label': row['true_label'], 'predicted_label': row['predicted_label'], 'outcome': row['outcome'],
        'status': row['status'], 'model_reasoning_summary': row.get('reasoning',''),
        'available_map_evidence': int(cov['map_evidence_total']), 'selected_evidence': selected_n,
        'selected_evidence_rate': selected_rate,
        'candidate_chains': int(ch['candidate_chain_count']),
        'full_four_group_co_presence': bool(ch['has_source_transform_action_destination']),
        'read_file_calls': int(nav['read_file_calls']), 'read_file_failed': int(nav['read_file_failed']),
        'candidate_failure_mode_NOT_VERIFIED': candidate,
        'why_this_candidate': rationale,
        'analyst_confirmed_failure_mode': '',
        'analyst_notes': '',
        'recommended_harness_change_hypothesis': '',
    })
failure_df = pd.DataFrame(review_rows)

## 7. PACKAGE-LEVEL MASTER TABLE + SUMMARY TABLES

In [ ]:
master_df = (package_df.merge(coverage_df, on=['sample_id','archive','ecosystem','true_label','predicted_label','outcome'], how='left', suffixes=('','_coverage'))
             .merge(navigation_df.drop(columns=['ecosystem','outcome','status'], errors='ignore'), on=['sample_id','archive'], how='left', suffixes=('','_nav'))
             .merge(chain_df.drop(columns=['archive','outcome'], errors='ignore'), on='sample_id', how='left', suffixes=('','_chain')))

outcome_order = ['TP','FP','FN','TN']
summary_rows = []
for outcome in outcome_order:
    sub = master_df[master_df['outcome'] == outcome]
    if sub.empty: continue
    summary_rows.append({
        'outcome': outcome, 'n_packages': len(sub),
        'mean_map_evidence': sub['map_evidence_total'].mean(),
        'mean_candidate_chains': sub['map_candidate_chains'].mean(),
        'mean_selected_evidence_count': sub['evidence_count'].mean(),
        'mean_scanned_bytes': sub['scanned_bytes'].mean(),
        'mean_dossier_chars': sub['dossier_chars'].mean(),
        'mean_api_requests': sub['api_requests'].mean(),
        'mean_total_tokens': sub['total_tokens'].mean(),
        'mean_tool_errors': sub['tool_errors'].mean(),
        'full_four_group_rate': sub['has_source_transform_action_destination'].mean(),
    })
outcome_summary_df = pd.DataFrame(summary_rows)

## 8. EXPORT REVIEW WORKBOOK

In [ ]:
readme_df = pd.DataFrame([
    ['Purpose', 'Diagnose H8.1 model/harness behavior on the fixed 24-package regression set.'],
    ['Scope warning', 'N=24 fixed regression set. Results are diagnostic and not independent external validation.'],
    ['Coverage distinction', 'The model receives both the dossier and get_behavior_map output. A fact whose file is absent from the dossier may still be exposed through the behavior map.'],
    ['Selection', 'selected_by_model indicates evidence IDs explicitly submitted by the model; it is not proof that the model semantically understood the fact.'],
    ['Chain caveat', 'Group co-presence and candidate_chains are static heuristics; same-file association is not proof of dataflow.'],
    ['Trace caveat', 'Token totals are summed from recorded API request usage. Tool errors count only explicit failure outcomes; inspect tool_outcomes_compact for details.'],
    ['Failure taxonomy', 'Candidate failure modes are heuristic prompts for manual review, not automatic diagnoses. Fill analyst_confirmed_failure_mode and analyst_notes.'],
    ['Recommended review order', 'Start with FN/FP rows in failure_taxonomy; cross-check coverage_facts, chain_reconstruction, navigation, then full chat/dossier in ZIP.'],
    ['Source files', f"{EDA_JSON.name}; {RUN_ZIP.name}"],
    ['Harness configuration', f"model={config.get('model')}; context={config.get('context_tokens')}; max_tokens={config.get('max_tokens')}; thinking={config.get('lm_studio_thinking_setting')} (not controlled by harness)."],
], columns=['item','description'])

with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
    readme_df.to_excel(writer, sheet_name='README', index=False)
    master_df.to_excel(writer, sheet_name='package_overview', index=False)
    outcome_summary_df.to_excel(writer, sheet_name='outcome_summary', index=False)
    coverage_df.to_excel(writer, sheet_name='coverage_summary', index=False)
    coverage_facts_df.to_excel(writer, sheet_name='coverage_facts', index=False)
    util_summary.to_excel(writer, sheet_name='utilization_by_outcome', index=False)
    util_by_package.to_excel(writer, sheet_name='utilization_by_package', index=False)
    chain_df.to_excel(writer, sheet_name='chain_reconstruction', index=False)
    navigation_df.to_excel(writer, sheet_name='navigation_trace', index=False)
    failure_df.to_excel(writer, sheet_name='failure_taxonomy', index=False)
    evidence_df.to_excel(writer, sheet_name='raw_evidence_registry', index=False)

# Basic usability formatting
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
wb = load_workbook(OUTPUT_XLSX)
for ws in wb.worksheets:
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions
    for cell in ws[1]:
        cell.font = Font(bold=True, color='FFFFFF')
        cell.fill = PatternFill('solid', fgColor='243B53')
        cell.alignment = Alignment(wrap_text=True, vertical='center')
    ws.row_dimensions[1].height = 32
    for col_cells in ws.columns:
        col_letter = get_column_letter(col_cells[0].column)
        max_len = max((len(str(c.value)) if c.value is not None else 0) for c in col_cells[:100])
        ws.column_dimensions[col_letter].width = min(max(max_len + 2, 12), 48)
    for row_cells in ws.iter_rows(min_row=2):
        for cell in row_cells:
            cell.alignment = Alignment(vertical='top', wrap_text=True)
# Highlight editable analyst columns
ws = wb['failure_taxonomy']
for header in ['analyst_confirmed_failure_mode','analyst_notes','recommended_harness_change_hypothesis']:
    idx = [c.value for c in ws[1]].index(header) + 1
    for r in range(2, ws.max_row + 1):
        ws.cell(r, idx).fill = PatternFill('solid', fgColor='FFF2CC')
wb.save(OUTPUT_XLSX)

print(f'Workbook written: {OUTPUT_XLSX.resolve()}')
print(f'Packages={len(package_df)}, coverage facts={len(coverage_facts_df)}, evidence rows={len(evidence_df)}, FN/FP review rows={len(failure_df)}')
print(outcome_summary_df.to_string(index=False))

Workbook written: /Users/manul-chik/omcb-work/project/project/analysis/H8_1_behavior_EDA_review.xlsx
Packages=24, coverage facts=187, evidence rows=187, FN/FP review rows=7
outcome  n_packages  mean_map_evidence  mean_candidate_chains  mean_selected_evidence_count  mean_scanned_bytes  mean_dossier_chars  mean_api_requests  mean_total_tokens  mean_tool_errors  full_four_group_rate
     TP           5          10.400000               1.000000                           7.0            32429.80         3550.000000           3.600000        9320.800000          0.200000              0.600000
     FN           7          13.000000               1.000000                           5.0             7202.00         3756.857143           3.285714        8622.857143          0.000000              0.857143
     TN          12           3.666667               0.333333                           1.0           210387.25         4402.333333           3.416667        8047.583333          0.083333          

## Выводы по блоку 1: поведение модели (H8.1, N=24)

**Результат прогона.** TP=5, FP=0, FN=7, TN=12; F1=0.588, precision=1.00, recall=0.42.
Покрытие 24/24. Модель ни разу не ошиблась в сторону malicious и пропустила больше половины
вредоносных пакетов. Набор существовал для диагностики.

**1. Проблема не в доступе к фактам.** У FN фактов в карте поведения в среднем больше, чем у TP
(13.0 против 10.4), а все четыре группы (источник, преобразование, действие, адрес) есть
у 86% FN и 60% TP. Модель получала нужные сведения и всё равно выносила benign.

**2. Сбой заметен уже на шаге анализа.** Примерно в 8 из 24 случаев вместо вывода стоит
заглушка: «Need more context», повтор задания или предупреждения из карты. Протокол принимает такой
ответ, поэтому `analysis_accepted` не означает, что анализ был выполнен.

**3. Выбор Evidence ID не отражает понимания.** В 7 из 8 вредоносных пакетов, где выбрано
8 ID, модель взяла первые 8 подряд. Лимит и порядок ID отсекали адреса назначения, поэтому
«низкая доля выбранных destinations» был артефактом интерфейса, а не наблюдением о модели.
Все 10 ответов с пустым `evidence_ids` закончились вердиктом benign.

**4. Вердикт выносился почти вслепую.** Финальный запрос содержал только короткий `reasoning`
и выбранные факты: без карты, без определения малвари и без скрытых рассуждений.

**5. Вердикт не определяется набором фактов.** Пакеты с почти одинаковыми картами получили
разные метки: `ajv` и `anf-core-react` верно, `am-alerter` и `com-unity` нет;
`requests-dm` и `SeleniumWebdriver` тоже разошлись.

**6. Наблюдается рационализация.** Модель описывает цепочку и объясняет её законной целью, которой
никто не заявлял. `brbsainath`: POST на webhook «consistent with the package's purpose», хотя
описание пакета пустое. `com-unity`: «legitimate» запрос на webhook/OAST.

**7. Часть ошибок идёт от детекторов.** В `angieslist` карта не нашла сеть из-за
`http = require('https')` и сборки адреса через `.join`, и модель последовала за картой.

**8. Объяснения слабее вердиктов.** Содержательный разбор есть примерно у 1 из 5 TP (`ajv`);
`qaqaqazzz` опирается на описание пакета, а у остальных трёх заглушка или план действий.

**Гипотезы для следующих экспериментов:** вердикту нужен полный контекст; заглушки надо отслеживать; выбор ID моделью надо заменить; детекторам нужны адреса и связи между файлами; а точная точка сбоя останется неизвестной, пока не сохраняются скрытые рассуждения.



## Блок 2— структура benchmark 


In [ ]:

# Manifest loading, archive resolution, static structure scan and shortcut audit
# ============================================================================

from pathlib import Path
import re
import tarfile
import zipfile
import collections
import warnings

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
)



OMCBENCH_ROOT = Path("FULL_PATH/omcb-work/omcb/")
MANIFEST_RELATIVE = Path("results") / "manifest.csv"

COLUMN_OVERRIDES = {
    "archive": None,
    "label": None,
    "ecosystem": None,
}


ROOT = OMCBENCH_ROOT.expanduser().resolve()

if not ROOT.is_dir():
    raise FileNotFoundError(
        f"OMCBENCH_ROOT не существует или не является директорией: {ROOT}"
    )

MANIFEST_PATH = ROOT / MANIFEST_RELATIVE

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Manifest не найден: {MANIFEST_PATH}"
    )

manifest_raw = pd.read_csv(MANIFEST_PATH)

print(f"OMCBench root: {ROOT}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"Manifest shape: {manifest_raw.shape}")
print("Manifest columns:", manifest_raw.columns.tolist())

if manifest_raw.empty:
    raise ValueError("Manifest пуст.")

if manifest_raw.shape[1] != 3:
    warnings.warn(
        f"Ожидалось 3 колонки в manifest, найдено {manifest_raw.shape[1]}."
    )


# mapping

def _norm_col(value):
    """Normalize column names for alias matching."""
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


ALIASES = {
    "archive": [
        "archive",
        "archivepath",
        "archive_path",
        "archivefile",
        "packagearchive",
        "filepath",
        "filename",
        "file",
        "path",
        "artifact",
        "artifactpath",
        "samplepath",
        "foldername",  
    ],

    "label": [
        "label",
        "labeltrue",   
        "class",
        "groundtruth",
        "target",
        "verdict",
        "malicious",
        "ismalicious",
        "is_malicious",
    ],

    "ecosystem": [
        "ecosystem",
        "packageecosystem",
        "language",
        "packagetype",
        "registry",
        "source",
    ],

    "package": [
        "package",
        "packagename",
        "project",
        "projectname",
        "name",
        "distribution",
    ],

    "version": [
        "version",
        "packageversion",
        "release",
        "releaseversion",
    ],
}

column_lookup = {
    _norm_col(column): column
    for column in manifest_raw.columns
}

column_map = {}

for key, aliases in ALIASES.items():

    override = COLUMN_OVERRIDES.get(key)

    if override is not None:

        if override not in manifest_raw.columns:
            raise KeyError(
                f"COLUMN_OVERRIDES[{key!r}]={override!r} "
                f"отсутствует в manifest. "
                f"Доступные колонки: {manifest_raw.columns.tolist()}"
            )

        column_map[key] = override
        continue

    matches = [
        column_lookup[_norm_col(alias)]
        for alias in aliases
        if _norm_col(alias) in column_lookup
    ]

    matches = list(dict.fromkeys(matches))

    if len(matches) > 1:
        raise ValueError(
            f"Неоднозначное сопоставление для {key!r}: {matches}. "
            f"Укажите COLUMN_OVERRIDES."
        )

    if len(matches) == 1:
        column_map[key] = matches[0]


if "archive" not in column_map or "label" not in column_map:

    raise ValueError(
        "Не удалось определить обязательные колонки archive и label.\n"
        f"Найдено сопоставление: {column_map}\n"
        f"Доступные колонки: {manifest_raw.columns.tolist()}\n"
        "Укажите реальные названия в COLUMN_OVERRIDES."
    )


if column_map["archive"] == column_map["label"]:
    raise ValueError("Колонки archive и label не могут совпадать.")

if (
    "ecosystem" in column_map
    and column_map["ecosystem"] in {
        column_map["archive"],
        column_map["label"],
    }
):
    raise ValueError("Колонка ecosystem должна отличаться от archive и label.")


print("\nColumn mapping:")
display(pd.DataFrame(
    [
        {"field": key, "manifest_column": value}
        for key, value in column_map.items()
    ]
))


# normalise

def _normalize_label(value):

    if pd.isna(value):
        raise ValueError("Обнаружена пустая метка.")

    if isinstance(value, (bool, np.bool_)):
        return int(value)

    if isinstance(value, (int, np.integer, float, np.floating)):

        if value in (0, 1):
            return int(value)

    value = str(value).strip().lower()

    malicious_values = {
        "malicious",
        "malware",
        "1",
        "true",
        "yes",
        "bad",
    }

    benign_values = {
        "benign",
        "clean",
        "0",
        "false",
        "no",
        "good",
    }

    if value in malicious_values:
        return 1

    if value in benign_values:
        return 0

    raise ValueError(
        f"Неизвестное значение метки: {value!r}"
    )


def _normalize_ecosystem(value, archive_value=""):

    if pd.notna(value):
        value = str(value).strip().lower()
    else:
        value = ""

    if value in {"npm", "javascript", "js", "node", "nodejs"}:
        return "npm"

    if value in {"pypi", "python", "py"}:
        return "pypi"

    if value:
        return value

    # Инференс только по однозначным форматам.
    archive = str(archive_value).strip().lower()

    if archive.endswith((".whl", ".tar.gz")):
        return "pypi"

    if archive.endswith(".tgz"):
        return "npm"

    # ZIP сам по себе не позволяет определить экосистему.
    return "unknown"



# ============================================================================

manifest_df = pd.DataFrame(index=manifest_raw.index)

manifest_df["manifest_row"] = np.arange(len(manifest_raw))

archive_values = manifest_raw[column_map["archive"]]

if archive_values.isna().any():
    raise ValueError(
        f"В archive-колонке найдено {int(archive_values.isna().sum())} пропусков."
    )

manifest_df["archive_ref"] = archive_values.astype(str).str.strip()

if manifest_df["archive_ref"].eq("").any():
    raise ValueError("В archive-колонке найдены пустые строки.")

manifest_df["label"] = (
    manifest_raw[column_map["label"]]
    .map(_normalize_label)
    .astype("int64")
)

manifest_df["label_name"] = (
    manifest_df["label"]
    .map({0: "benign", 1: "malicious"})
)

if "ecosystem" in column_map:

    manifest_df["ecosystem"] = [
        _normalize_ecosystem(ecosystem, archive)
        for ecosystem, archive in zip(
            manifest_raw[column_map["ecosystem"]],
            manifest_df["archive_ref"],
        )
    ]

else:

    manifest_df["ecosystem"] = [
        _normalize_ecosystem(None, archive)
        for archive in manifest_df["archive_ref"]
    ]

if "package" in column_map:

    manifest_df["package_name"] = (
        manifest_raw[column_map["package"]]
        .fillna("")
        .astype(str)
        .str.strip()
    )

else:

    manifest_df["package_name"] = ""


if "version" in column_map:

    manifest_df["version"] = (
        manifest_raw[column_map["version"]]
        .fillna("")
        .astype(str)
        .str.strip()
    )

else:

    manifest_df["version"] = ""


print("\nNormalized manifest:")
print("Rows:", len(manifest_df))
print("Ecosystems:", manifest_df["ecosystem"].value_counts(dropna=False).to_dict())
print("Labels:", manifest_df["label_name"].value_counts(dropna=False).to_dict())


unknown_ecosystems = sorted(
    set(manifest_df["ecosystem"]) - {"npm", "pypi"}
)

if unknown_ecosystems:

    raise ValueError(
        "Не удалось определить экосистему для некоторых записей: "
        f"{unknown_ecosystems}. "
        "Проверьте ecosystem-колонку manifest."
    )




def _normalize_path_string(value):
    return str(value).replace("\\", "/").strip()


def _resolve_direct_path(reference):

    path = Path(reference).expanduser()

    if path.is_absolute():

        return path.resolve() if path.is_file() else None

    candidates = [
        ROOT / path,
        ROOT / "archives" / path,
        ROOT / "packages" / path,
        ROOT / "data" / path,
        ROOT / "samples" / path,
    ]

    for candidate in candidates:

        if candidate.is_file():
            return candidate.resolve()

    return None


# Сначала пробуем разрешить все пути напрямую.
resolved = [
    _resolve_direct_path(reference)
    for reference in manifest_df["archive_ref"]
]


# Индексируем архивы только если остались неразрешённые пути.
missing_positions = [
    index
    for index, path in enumerate(resolved)
    if path is None
]


if missing_positions:

    print(
        f"\nDirect path resolution: "
        f"{len(resolved) - len(missing_positions)}/{len(resolved)}"
    )

    print("Building archive index for unresolved paths...")

    archive_index = collections.defaultdict(list)

    archive_patterns = (
        "*.tgz",
        "*.tar.gz",
        "*.whl",
        "*.zip",
    )

    for pattern in archive_patterns:

        for path in ROOT.rglob(pattern):

            if path.is_file():
                archive_index[path.name].append(path.resolve())


    for index in missing_positions:

        reference = manifest_df.iloc[index]["archive_ref"]
        reference_name = Path(reference).name

        matches = archive_index.get(reference_name, [])

        if len(matches) == 1:

            resolved[index] = matches[0]

        elif len(matches) > 1:

            ref_normalized = _normalize_path_string(reference).lstrip("./")

            relative_matches = []

            for candidate in matches:

                try:
                    relative_path = _normalize_path_string(
                        candidate.relative_to(ROOT)
                    )
                except ValueError:
                    relative_path = _normalize_path_string(candidate)

                if (
                    relative_path == ref_normalized
                    or relative_path.endswith("/" + ref_normalized)
                ):
                    relative_matches.append(candidate)

            if len(relative_matches) == 1:
                resolved[index] = relative_matches[0]


manifest_df["archive_path"] = resolved



missing_paths = manifest_df["archive_path"].isna()

print(
    f"\nResolved archive paths: "
    f"{int((~missing_paths).sum())}/{len(manifest_df)}"
)

if missing_paths.any():

    print("\nUnresolved archive references:")

    display(
        manifest_df.loc[
            missing_paths,
            ["archive_ref", "ecosystem", "label_name"],
        ].head(30)
    )

    raise FileNotFoundError(
        f"{int(missing_paths.sum())} архивов не удалось найти. "
        "Не продолжаем анализ на неполной выборке."
    )


if manifest_df["archive_path"].duplicated().any():

    print("\nWARNING: одинаковые физические архивы встречаются более одного раза.")

    display(
        manifest_df.loc[
            manifest_df["archive_path"].duplicated(keep=False),
            ["archive_ref", "archive_path", "ecosystem", "label_name"],
        ].head(30)
    )


if manifest_df.duplicated(["archive_ref"]).any():

    print("\nWARNING: повторяющиеся archive_ref:")

    display(
        manifest_df.loc[
            manifest_df["archive_ref"].duplicated(keep=False),
            ["archive_ref", "ecosystem", "label_name"],
        ].head(30)
    )




SOURCE_EXTS = {
    ".py",
    ".js",
    ".mjs",
    ".cjs",
    ".ts",
    ".tsx",
    ".jsx",
}

TEXT_EXTS = SOURCE_EXTS | {
    ".json",
    ".toml",
    ".yaml",
    ".yml",
    ".ini",
    ".cfg",
    ".md",
    ".txt",
}


def _archive_format(name):

    name = str(name).lower()

    for suffix, archive_type in [
        (".tar.gz", "tar.gz"),
        (".tgz", "tgz"),
        (".whl", "whl"),
        (".zip", "zip"),
    ]:

        if name.endswith(suffix):
            return archive_type

    return "unknown"


def _member_ext(name):

    base = name.rstrip("/").rsplit("/", 1)[-1]

    if (
        "." not in base
        or (base.startswith(".") and base.count(".") == 1)
    ):
        return "[no_ext]"

    return "." + base.rsplit(".", 1)[-1].lower()

# zip scanning

def _scan_zip(path):

    with zipfile.ZipFile(path, "r") as archive:

        infos = archive.infolist()

        files = [
            item
            for item in infos
            if not item.is_dir()
        ]

        directories = [
            item
            for item in infos
            if item.is_dir()
        ]

        names = [
            item.filename.replace("\\", "/").strip("/")
            for item in infos
            if item.filename.strip("/")
        ]

        uncompressed_size = sum(
            item.file_size
            for item in files
        )

        compressed_members = sum(
            item.compress_size
            for item in files
        )

        file_names = [
            item.filename
            for item in files
        ]

        return (
            names,
            len(files),
            len(directories),
            uncompressed_size,
            compressed_members,
            file_names,
        )


# tar scanning

def _scan_tar(path):

    with tarfile.open(path, mode="r:*") as archive:

        members = archive.getmembers()

        files = [
            item
            for item in members
            if item.isfile()
        ]

        directories = [
            item
            for item in members
            if item.isdir()
        ]

        names = [
            item.name.replace("\\", "/").strip("/")
            for item in members
            if item.name.strip("/")
        ]

        uncompressed_size = sum(
            item.size
            for item in files
        )

        file_names = [
            item.name
            for item in files
        ]

        return (
            names,
            len(files),
            len(directories),
            uncompressed_size,
            None,
            file_names,
        )


# one scanning

def _scan_one(row):

    path = Path(row["archive_path"])

    # Формат - по фактическому пути к архиву.
    archive_type = _archive_format(path.name)

    try:

        if archive_type in {"zip", "whl"}:

            (
                names,
                n_files,
                n_dirs,
                uncompressed_size,
                compressed_members,
                file_names,
            ) = _scan_zip(path)

        elif archive_type in {"tgz", "tar.gz"}:

            (
                names,
                n_files,
                n_dirs,
                uncompressed_size,
                compressed_members,
                file_names,
            ) = _scan_tar(path)

        else:

            raise ValueError(
                f"Unsupported archive format: {archive_type}"
            )


        clean_names = [
            name
            for name in names
            if name
        ]

        roots = sorted({
            name.split("/", 1)[0]
            for name in clean_names
        })

        root_file_count = sum(
            1
            for name in file_names
            if "/" not in name.strip("/")
        )

        if len(roots) == 1 and root_file_count == 0:

            root_layout = "single_root_dir"

        elif root_file_count > 0:

            root_layout = "root_files_or_mixed"

        else:

            root_layout = "multi_root"


        root_name = roots[0] if len(roots) == 1 else ""

        lower_files = [
            name.lower()
            for name in file_names
        ]

        lower_names = [
            name.lower()
            for name in clean_names
        ]

        extension_counts = collections.Counter(
            _member_ext(name)
            for name in file_names
        )

        source_count = sum(
            extension_counts[extension]
            for extension in SOURCE_EXTS
        )

        text_count = sum(
            extension_counts[extension]
            for extension in TEXT_EXTS
        )

        has_dist_info = int(
            any(
                ".dist-info/" in ("/" + name)
                for name in lower_names
            )
        )

        has_package_wrapper = int(
            any(
                name == "package" or name.startswith("package/")
                for name in lower_names
            )
        )

        archive_bytes = path.stat().st_size

        compression_ratio = (
            uncompressed_size / archive_bytes
            if archive_bytes > 0
            else np.nan
        )

        return {

            "archive_bytes": archive_bytes,

            "uncompressed_bytes": uncompressed_size,

            "compression_ratio": compression_ratio,

            "archive_format": archive_type,

            "n_files": n_files,

            "n_dirs": n_dirs,

            "n_members": n_files + n_dirs,

            "root_file_count": root_file_count,

            "top_level_count": len(roots),

            "root_layout": root_layout,

            "single_root_name": root_name,

            "has_package_wrapper": has_package_wrapper,

            "has_dist_info": has_dist_info,

            "n_source_files": source_count,

            "n_text_files": text_count,

            "n_py_files": int(extension_counts[".py"]),

            "n_js_files": int(
                extension_counts[".js"]
                + extension_counts[".mjs"]
                + extension_counts[".cjs"]
            ),

            "n_json_files": int(extension_counts[".json"]),

            "member_extensions": ";".join(
                f"{key}:{value}"
                for key, value in sorted(extension_counts.items())
            ),

            "scan_error": "",

        }

    except Exception as exc:

        return {
            "scan_error": f"{type(exc).__name__}: {exc}"
        }




print("\nStarting static archive metadata scan...")

scan_rows = [
    _scan_one(row)
    for row in manifest_df.to_dict("records")
]

scan_df = pd.DataFrame(scan_rows)

archive_df = pd.concat(
    [
        manifest_df.reset_index(drop=True),
        scan_df.reset_index(drop=True),
    ],
    axis=1,
)

scan_errors = (
    archive_df["scan_error"]
    .fillna("")
    .ne("")
)

print(
    f"Scanned: {len(archive_df)} | "
    f"scan errors: {int(scan_errors.sum())}"
)

if scan_errors.any():

    display(
        archive_df.loc[
            scan_errors,
            ["archive_ref", "archive_path", "scan_error"],
        ].head(30)
    )

    raise RuntimeError(
        "Есть ошибки чтения архивов. "
        "Сначала разберите их; не анализируйте неполный набор."
    )



if len(archive_df) != len(manifest_raw):

    raise AssertionError(
        "Количество строк после сканирования не совпадает с manifest."
    )


if archive_df["label"].isna().any():

    raise ValueError("В итоговом датасете обнаружены пустые метки.")


if archive_df["ecosystem"].isna().any():

    raise ValueError("В итоговом датасете обнаружены пустые экосистемы.")


print("\nDataset integrity:")
print(f"Total samples: {len(archive_df)}")
print(f"Unique archive paths: {archive_df['archive_path'].nunique()}")
print(f"Scan errors: {int(scan_errors.sum())}")



print("\nClass balance:")

display(
    pd.crosstab(
        archive_df["ecosystem"],
        archive_df["label_name"],
        margins=True,
    )
)



print("\nArchive format × label:")

display(
    pd.crosstab(
        [
            archive_df["ecosystem"],
            archive_df["archive_format"],
        ],
        archive_df["label_name"],
        margins=True,
    )
)



archive_df["archive_kb"] = (
    archive_df["archive_bytes"] / 1024
)

size_summary = (
    archive_df
    .groupby(
        ["ecosystem", "label_name"]
    )["archive_kb"]
    .agg(
        n="count",
        median="median",
        mean="mean",
        max="max",
    )
    .round(2)
)

print("\nArchive sizes, KB:")

display(size_summary)


# ============================================================================


structure_cols = [

    "archive_bytes",
    "uncompressed_bytes",
    "compression_ratio",

    "n_files",
    "n_dirs",
    "n_members",

    "root_file_count",
    "top_level_count",

    "has_package_wrapper",
    "has_dist_info",

    "n_source_files",
    "n_text_files",

    "n_py_files",
    "n_js_files",
    "n_json_files",

]

structure_summary = (
    archive_df
    .groupby(
        ["ecosystem", "label_name"]
    )[structure_cols]
    .agg(
        ["count", "median", "mean", "max"]
    )
)

print("\nStructural feature summary:")

display(structure_summary.round(2))


# ============================================================================

print("\nRoot layout × label:")

display(
    pd.crosstab(
        [
            archive_df["ecosystem"],
            archive_df["root_layout"],
        ],
        archive_df["label_name"],
        margins=True,
    )
)


# ============================================================================

for column in [
    "has_package_wrapper",
    "has_dist_info",
]:

    print(f"\n{column} × label:")

    display(
        pd.crosstab(
            [
                archive_df["ecosystem"],
                archive_df[column],
            ],
            archive_df["label_name"],
            margins=True,
        )
    )


# ============================================================================


#
# npm:
#     .zip -> malicious
#
# PyPI:
#     anything except .whl -> malicious
#



def format_rule(row):

    if row["ecosystem"] == "npm":

        return int(
            row["archive_format"] == "zip"
        )

    if row["ecosystem"] == "pypi":

        return int(
            row["archive_format"] != "whl"
        )

    raise ValueError(
        f"Unknown ecosystem: {row['ecosystem']}"
    )


archive_df["format_rule_pred"] = (
    archive_df.apply(format_rule, axis=1)
)

archive_df["y_true"] = (
    archive_df["label"].astype(int)
)

archive_df["format_rule_correct"] = (
    archive_df["format_rule_pred"]
    == archive_df["y_true"]
)


def _metric_row(name, subset):

    y_true = subset["y_true"].astype(int)

    y_pred = subset["format_rule_pred"].astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    return {

        "scope": name,

        "n": len(subset),

        "TP": tp,

        "FP": fp,

        "FN": fn,

        "TN": tn,

        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),

        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),

        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),

        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),

    }


rule_rows = [
    _metric_row("all", archive_df)
]

for ecosystem in sorted(
    archive_df["ecosystem"].unique()
):

    subset = archive_df[
        archive_df["ecosystem"] == ecosystem
    ]

    rule_rows.append(
        _metric_row(ecosystem, subset)
    )


format_rule_results = pd.DataFrame(rule_rows)

print("\nFormat shortcut performance:")

display(
    format_rule_results.round(4)
)


# ============================================================================
# 21. ARCHIVE SIZE QUANTILES
# ============================================================================

archive_df["archive_mb"] = (
    archive_df["archive_bytes"] / (1024 ** 2)
)

size_quantiles = (
    archive_df
    .groupby(
        ["ecosystem", "label_name"]
    )["archive_mb"]
    .quantile(
        [0.0, 0.25, 0.5, 0.75, 0.9, 1.0]
    )
    .unstack()
    .rename(
        columns={
            0.0: "min",
            0.25: "q25",
            0.5: "median",
            0.75: "q75",
            0.9: "q90",
            1.0: "max",
        }
    )
)

print("\nArchive size quantiles, MB:")

display(
    size_quantiles.round(4)
)


# ============================================================================

print("\nEDA objects created:")

print("manifest_raw       — original manifest")
print("manifest_df        — normalized manifest")
print("archive_df         — manifest + archive metadata")
print("size_summary      — archive size summary")
print("structure_summary — structural feature summary")
print("format_rule_results — format shortcut metrics")
print("size_quantiles    — archive size quantiles")

print("\nEDA 9.1–9.2 completed successfully.")

OMCBench root: /Users/manul-chik/omcb-work/omcb
Manifest: /Users/manul-chik/omcb-work/omcb/results/manifest.csv
Manifest shape: (800, 3)
Manifest columns: ['folder_name', 'ecosystem', 'label_true']

Column mapping:


,field,manifest_column
0,archive,folder_name
1,label,label_true
2,ecosystem,ecosystem



Normalized manifest:
Rows: 800
Ecosystems: {'pypi': 400, 'npm': 400}
Labels: {'benign': 400, 'malicious': 400}

Resolved archive paths: 800/800

Starting static archive metadata scan...
Scanned: 800 | scan errors: 0

Dataset integrity:
Total samples: 800
Unique archive paths: 800
Scan errors: 0

Class balance:


label_name,benign,malicious,All
ecosystem,,,
npm,200,200,400
pypi,200,200,400
All,400,400,800



Archive format × label:


label_name                benign  malicious  All
ecosystem archive_format                        
npm       tgz                200          0  200
          zip                  0        200  200
pypi      tar.gz              23        155  178
          whl                177          5  182
          zip                  0         40   40
All                          400        400  800

... (11 outputs left out)

,scope,n,TP,FP,FN,TN,precision,recall,F1,accuracy
0,all,800,395,23,5,377,0.9450,0.9875,0.9658,0.965
1,npm,400,200,0,0,200,1.0000,1.0000,1.0000,1.000
2,pypi,400,195,23,5,177,0.8945,0.9750,0.9330,0.930



Archive size quantiles, MB:


min     q25  median     q75     q90      max
ecosystem label_name                                                 
npm       benign      0.0008  0.0047  0.0093  0.0291  0.1216   9.5155
          malicious   0.0005  0.0009  0.0013  0.0025  0.0245   2.1949
pypi      benign      0.0029  0.0147  0.0378  0.1414  0.6029  28.0952
          malicious   0.0009  0.0021  0.0052  0.0256  0.1004   6.4886


EDA objects created:
manifest_raw       — original manifest
manifest_df        — normalized manifest
archive_df         — manifest + archive metadata
size_summary      — archive size summary
structure_summary — structural feature summary
format_rule_results — format shortcut metrics
size_quantiles    — archive size quantiles

EDA 9.1–9.2 completed successfully.


## Блок 2.2 — корреляции и ассоциации с меткой

Для числовых признаков — Spearman и point-biserial; для категориальных — Cramér's V. Корреляция не доказывает причинность; pooled-результаты дополнительно рассматриваем по экосистемам.

In [ ]:

from scipy.stats import spearmanr, pointbiserialr, chi2_contingency

numeric_features = [
    "archive_bytes", "uncompressed_bytes", "compression_ratio",
    "n_files", "n_dirs", "n_members", "root_file_count", "top_level_count",
    "has_package_wrapper", "has_dist_info", "n_source_files", "n_text_files",
    "n_py_files", "n_js_files", "n_json_files",
]
corr_rows = []
for col in numeric_features:
    x = pd.to_numeric(archive_df[col], errors="coerce")
    valid = x.notna() & archive_df["y_true"].notna()
    if valid.sum() < 3 or x[valid].nunique() < 2:
        continue
    sp = spearmanr(x[valid], archive_df.loc[valid, "y_true"])
    pb = pointbiserialr(archive_df.loc[valid, "y_true"], x[valid])
    corr_rows.append({
        "feature": col, "n": int(valid.sum()),
        "spearman_r": float(sp.statistic), "spearman_p": float(sp.pvalue),
        "point_biserial_r": float(pb.statistic), "point_biserial_p": float(pb.pvalue),
    })
numeric_correlations = pd.DataFrame(corr_rows)
if not numeric_correlations.empty:
    numeric_correlations["abs_spearman"] = numeric_correlations["spearman_r"].abs()
    numeric_correlations = numeric_correlations.sort_values("abs_spearman", ascending=False)
display(numeric_correlations.round(4))

def cramers_v(x, y):
    tab = pd.crosstab(x, y)
    if tab.empty or min(tab.shape) < 2:
        return np.nan
    chi2 = chi2_contingency(tab, correction=False)[0]
    n = tab.to_numpy().sum()
    phi2 = chi2 / n
    r, k = tab.shape
    # Bias-corrected Cramer's V (Bergsma correction)
    if n <= 1:
        return np.nan
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    denom = min(kcorr - 1, rcorr - 1)
    return math.sqrt(phi2corr / denom) if denom > 0 else np.nan

categorical_features = ["ecosystem", "archive_format", "root_layout",
                        "has_package_wrapper", "has_dist_info"]
cat_corr_rows = []
for col in categorical_features:
    vals = archive_df[col].fillna("[missing]").astype(str)
    cat_corr_rows.append({
        "feature": col,
        "n_levels": vals.nunique(),
        "cramers_v_with_label": cramers_v(vals, archive_df["label_name"]),
    })
categorical_associations = pd.DataFrame(cat_corr_rows).sort_values(
    "cramers_v_with_label", ascending=False
)
display(categorical_associations.round(4))

# Ecosystem-conditional size correlations to avoid interpreting pooled effects alone.
eco_corr_rows = []
for eco, sub in archive_df.groupby("ecosystem"):
    for col in ["archive_bytes", "uncompressed_bytes", "n_files", "n_source_files"]:
        x = pd.to_numeric(sub[col], errors="coerce")
        valid = x.notna()
        if valid.sum() >= 3 and x[valid].nunique() > 1:
            eco_corr_rows.append({
                "ecosystem": eco, "feature": col, "n": int(valid.sum()),
                "spearman_r": spearmanr(x[valid], sub.loc[valid, "y_true"]).statistic,
            })
ecosystem_correlations = pd.DataFrame(eco_corr_rows)
display(ecosystem_correlations.round(4))


,feature,n,spearman_r,spearman_p,point_biserial_r,point_biserial_p,abs_spearman
2,compression_ratio,800,-0.6115,0.0000,-0.5135,0.0000,0.6115
8,has_package_wrapper,800,-0.5716,0.0000,-0.5716,0.0000,0.5716
1,uncompressed_bytes,800,-0.5641,0.0000,-0.1235,0.0005,0.5641
6,root_file_count,800,0.5332,0.0000,0.4475,0.0000,0.5332
0,archive_bytes,800,-0.5133,0.0000,-0.1116,0.0016,0.5133
9,has_dist_info,800,-0.5129,0.0000,-0.5129,0.0000,0.5129
10,n_source_files,800,-0.5017,0.0000,-0.1163,0.0010,0.5017
3,n_files,800,-0.4326,0.0000,-0.0681,0.0540,0.4326
5,n_members,800,-0.3792,0.0000,-0.0661,0.0618,0.3792
11,n_text_files,800,-0.3772,0.0000,-0.1123,0.0015,0.3772


,feature,n_levels,cramers_v_with_label
1,archive_format,4,0.9343
2,root_layout,3,0.6125
3,has_package_wrapper,2,0.5708
4,has_dist_info,2,0.5120
0,ecosystem,2,0.0000


,ecosystem,feature,n,spearman_r
0,npm,archive_bytes,400,-0.5870
1,npm,uncompressed_bytes,400,-0.6211
2,npm,n_files,400,-0.5842
3,npm,n_source_files,400,-0.4228
4,pypi,archive_bytes,400,-0.5326
5,pypi,uncompressed_bytes,400,-0.5800
6,pypi,n_files,400,-0.5020
7,pypi,n_source_files,400,-0.6339


## Блок 2.3 — sklearn baseline и экспорт

Сравнивнение Dummy, Logistic Regression и Decision Tree на табличных признаках. Имя пакета, версия, путь и номер строки не используются как предикторы. Если есть повторяющиеся package names, применяется StratifiedGroupKFold. Результаты сохраняются в `results/OMCBench_full800_EDA.xlsx`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    confusion_matrix, classification_report
)

# Group-aware CV if multiple versions/rows refer to the same package.
groups = None
group_cv_used = False
if "package_name" in archive_df.columns:
    g = archive_df["package_name"].fillna("").astype(str).str.strip()
    valid_names = g.ne("") & ~g.str.lower().isin({"nan", "none", "unknown"})
    if valid_names.sum() == len(g) and g.nunique() < len(g):
        groups = g
        group_cv_used = True

y = archive_df["y_true"].astype(int).to_numpy()
if group_cv_used:
    try:
        from sklearn.model_selection import StratifiedGroupKFold
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
        splits = list(cv.split(archive_df, y, groups=groups))
        # Validate that each validation fold contains both classes.
        if any(len(np.unique(y[va])) < 2 for _, va in splits):
            raise ValueError("A validation fold has only one class.")
        print("CV: StratifiedGroupKFold(5), grouped by package_name")
    except Exception as exc:
        warnings.warn(f"Group-aware CV unavailable/invalid ({exc}); falling back to StratifiedKFold.")
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        splits = list(cv.split(archive_df, y))
        groups = None
else:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    splits = list(cv.split(archive_df, y))
    print("CV: StratifiedKFold(5); no repeated package names detected or package name unavailable.")

numeric_sets = {
    "size_only": ["archive_bytes"],
    "size_and_contents": ["archive_bytes", "uncompressed_bytes", "compression_ratio"],
    "structure_numeric": [
        "n_files", "n_dirs", "n_members", "root_file_count", "top_level_count",
        "has_package_wrapper", "has_dist_info", "n_source_files", "n_text_files",
        "n_py_files", "n_js_files", "n_json_files",
    ],
    "all_numeric": [
        "archive_bytes", "uncompressed_bytes", "compression_ratio",
        "n_files", "n_dirs", "n_members", "root_file_count", "top_level_count",
        "has_package_wrapper", "has_dist_info", "n_source_files", "n_text_files",
        "n_py_files", "n_js_files", "n_json_files",
    ],
}
categorical_sets = {
    "format_only": [
        "archive_format",
    ],

    "layout_only": [
        "ecosystem",
        "root_layout",
    ],

    "all_categorical": [
        "ecosystem",
        "archive_format",
        "root_layout",
    ],
}

feature_sets = {
    "size_only": (
        numeric_sets["size_only"],
        [],
    ),

    "format_only": (
        [],
        categorical_sets["format_only"],
    ),

    "structure_only": (
        numeric_sets["structure_numeric"],
        categorical_sets["layout_only"],
    ),

    "all_metadata": (
        numeric_sets["all_numeric"],
        categorical_sets["all_categorical"],
    ),
}

def make_preprocessor(num_cols, cat_cols, scale_numeric=True):
    transformers = []
    if num_cols:
        num_steps = [("imputer", SimpleImputer(strategy="median"))]
        if scale_numeric:
            num_steps.append(("scaler", StandardScaler()))
        transformers.append(("num", Pipeline(num_steps), num_cols))
    if cat_cols:
        transformers.append(("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols))
    return ColumnTransformer(transformers, remainder="drop")

results = []
prediction_frames = []
for feature_name, (num_cols, cat_cols) in feature_sets.items():
    cols = num_cols + cat_cols
    X = archive_df[cols].copy()
    pre = make_preprocessor(num_cols, cat_cols, scale_numeric=True)

    estimators = {
        "LogisticRegression": LogisticRegression(
            max_iter=3000, class_weight="balanced", C=1.0, random_state=42
        ),
        "DecisionTree": DecisionTreeClassifier(
            max_depth=4, min_samples_leaf=8, class_weight="balanced", random_state=42
        ),
    }
    for model_name, estimator in estimators.items():
        pipe = Pipeline([("prep", pre), ("model", estimator)])
        if groups is not None and group_cv_used:
            pred = cross_val_predict(pipe, X, y, cv=splits, groups=groups, method="predict")
        else:
            pred = cross_val_predict(pipe, X, y, cv=splits, method="predict")
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
        results.append({
            "feature_set": feature_name, "model": model_name, "n": len(y),
            "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "F1_malicious": f1_score(y, pred, zero_division=0),
            "accuracy": accuracy_score(y, pred),
        })
        prediction_frames.append(pd.DataFrame({
            "manifest_row": archive_df["manifest_row"],
            "archive_ref": archive_df["archive_ref"],
            "ecosystem": archive_df["ecosystem"],
            "y_true": y, "y_pred": pred,
            "feature_set": feature_name, "model": model_name,
        }))

# Dummy baseline using the same folds.
dummy_pred = np.zeros_like(y)
for _, va in splits:
    train_idx = np.setdiff1d(np.arange(len(y)), va)
    majority = int(pd.Series(y[train_idx]).mode().iloc[0])
    dummy_pred[va] = majority
tn, fp, fn, tp = confusion_matrix(y, dummy_pred, labels=[0, 1]).ravel()
results.append({
    "feature_set": "no_features", "model": "DummyMostFrequent", "n": len(y),
    "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    "precision": precision_score(y, dummy_pred, zero_division=0),
    "recall": recall_score(y, dummy_pred, zero_division=0),
    "F1_malicious": f1_score(y, dummy_pred, zero_division=0),
    "accuracy": accuracy_score(y, dummy_pred),
})

sklearn_results = pd.DataFrame(results).sort_values("F1_malicious", ascending=False)
sklearn_predictions = pd.concat(prediction_frames, ignore_index=True)
display(sklearn_results.round(4))


print("\nImportant: high CV F1 here measures label predictability from benchmark metadata,")
print("not malware-detection capability and not evidence of what the LLM actually used.")


BENCHMARK_EDA_XLSX = ROOT / "results" / "OMCBench_full800_EDA.xlsx"
BENCHMARK_EDA_XLSX.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(BENCHMARK_EDA_XLSX, engine="openpyxl") as writer:
    archive_df.to_excel(writer, sheet_name="package_structure", index=False)
    size_summary.reset_index().to_excel(writer, sheet_name="size_summary", index=False)
    format_rule_results.to_excel(writer, sheet_name="format_rule", index=False)
    size_quantiles.reset_index().to_excel(writer, sheet_name="size_quantiles", index=False)
    numeric_correlations.to_excel(writer, sheet_name="numeric_correlations", index=False)
    categorical_associations.to_excel(writer, sheet_name="categorical_associations", index=False)
    ecosystem_correlations.to_excel(writer, sheet_name="ecosystem_correlations", index=False)
    sklearn_results.to_excel(writer, sheet_name="sklearn_baselines", index=False)
    sklearn_predictions.to_excel(writer, sheet_name="sklearn_predictions", index=False)

print(f"Saved: {BENCHMARK_EDA_XLSX}")
print(f"Rows analyzed: {len(archive_df)}")
print(f"Class counts: {archive_df['label_name'].value_counts().to_dict()}")
print(f"Ecosystem counts: {archive_df['ecosystem'].value_counts().to_dict()}")


CV: StratifiedKFold(5); no repeated package names detected or package name unavailable.


,feature_set,model,n,TP,FP,FN,TN,precision,recall,F1_malicious,accuracy
7,all_metadata,DecisionTree,800,389,14,11,386,0.9653,0.9725,0.9689,0.9688
6,all_metadata,LogisticRegression,800,391,18,9,382,0.9560,0.9775,0.9666,0.9662
2,format_only,LogisticRegression,800,395,23,5,377,0.9450,0.9875,0.9658,0.9650
3,format_only,DecisionTree,800,395,23,5,377,0.9450,0.9875,0.9658,0.9650
4,structure_only,LogisticRegression,800,392,22,8,378,0.9469,0.9800,0.9631,0.9625
5,structure_only,DecisionTree,800,373,25,27,375,0.9372,0.9325,0.9348,0.9350
1,size_only,DecisionTree,800,237,38,163,362,0.8618,0.5925,0.7022,0.7488
0,size_only,LogisticRegression,800,370,313,30,87,0.5417,0.9250,0.6833,0.5712
8,no_features,DummyMostFrequent,800,0,0,400,400,0.0000,0.0000,0.0000,0.5000



Important: high CV F1 here measures label predictability from benchmark metadata,
not malware-detection capability and not evidence of what the LLM actually used.
Saved: /Users/manul-chik/omcb-work/omcb/results/OMCBench_full800_EDA.xlsx
Rows analyzed: 800
Class counts: {'benign': 400, 'malicious': 400}
Ecosystem counts: {'pypi': 400, 'npm': 400}
